# Evaluation
*Reyna Geluk, Stan Bakker, Mieke Fraters & Aryan Müller*

The evaluation runs on the algorithm without the additional boost to smaller game titles and publishers. This was to simply check how well the algorithm itself runs without any adjustments and additions, and to hopefully get some kind of an idea of its quality and workings, despite the rather sparse data it is trained on. The evaluation can be found in the last cell of this notebook, the section 5, whilst the first part is the main part of the algorithm. 

The evaluation begins by defining its parameters, a top-25 check depth, 25 similar users and apps, a minimum of 10 reviews per user, and a sample of 250. Then it filters the full dataset to users who meet that review threshold and have at least one positive review to hide, drawing a random sample from that pool. For each sampled user one positively reviewed game is selected at random and stored, after which every one of those hidden reviews is removed from the DataFrame, so the recommender cannot see them. The interaction matrix is then rebuilt from scratch on this cleaned data with fresh user and app indices to keep the matrix internally consistent, and a new KNN model is fitted on it, while the TF-IDF matrix from the main model is reused unchanged since game tags are unaffected by which reviews were removed. Evaluation specific versions of the three main helper functions are then defined, mirroring the main recommender exactly but operating on the cleaned matrix, and for each test user the recommender is run to produce a ranked list of 25 game recommendations, which is then checked for the presence of the hidden game. Finally, the results are aggregated into a hit rate, hits divided by successfully evaluated users, and printed alongside the average and median rank of the hidden game in cases where it was found. This is to give both an accuracy figure and a sense of how confidently the system surfaces the correct result when it does.

In [6]:
# Once im sure everything is ready and wont break this file and let it take ages to load, ill add my shabam of data cleaning and stuff in here.

In [8]:
#Imports
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random

In [9]:

df = pd.read_csv("C:/Program Files/allFilteredAggr7.csv") # Change dataset depending on what files we use!!!!

#UnicodeDecodeError: 'utf-8' codec can't decode byte 0x99 in position 22574: invalid start byte
#Bla Bla throw more code at it till we don't get that error
games_df =  pd.read_csv("games.csv", sep=";", encoding="latin-1") #change the name depending on what it is called. 
games_df = games_df.rename(columns={"AppID": "appid"})
#also please call the appid column tags_appid... please.... and tags tags_tag (　＾▽＾)

## Creating the recommender algoirhtm

In [12]:
# ═════════════════════════════════════════════════════════════════════════════
# 0. MY NOTES AND THOUGHTS
# ═════════════════════════════════════════════════════════════════════════════
#So I need to make:

# Collaborative filtering: Users like you also liked. Using players with similiar voting patterns and stuff. (User-User KNN)
# Content- based filtering: Games similar to ones you liked. We are gonna use the tags. (TF-IDF + Cosine Similarity)
# Hybrid recommender: combine the two from above
# I will just be figuring out what the dude did and copy it so that is why the code might be a bit weird? like wtf are all those cat things
# They make my head spin with the logic behind it XD oeh I am gonna use the fancy code barriers
# Oh yea I used printing inbetween to see what it was doing and if going wrong where


# ═════════════════════════════════════════════════════════════════════════════
# 1. MAKING THE CODE USABLE OR SOMETHING
# ═════════════════════════════════════════════════════════════════════════════

# Map users and apps to contiguous integer indices.
#So bassically what this part does. It grabs the information from the columns. Tells panda to see it as type "category" aka unique labels not just numbers
#And apparently cat.codes replaces the unique label with a small integer assigned in sorted order. So no more huge numbers? idk
# So .loc[:. "name"] is need or else pandas starts complaining "A value is trying to be set on a copy of a slice from a DataFrame." it legit told me
# To do it this way
df.loc[:, "user_idx"] = df["author_steamid"].astype("category").cat.codes
df.loc[:, "app_idx"]  = df["appid"].astype("category").cat.codes

# .cat.categories is the reverse lookup: a sorted array of the original IDs, where position i holds the original ID that was assigned cat.code i.
# We save this so we can convert back from matrix indices to real IDs later.
unique_user_ids = df["author_steamid"].astype("category").cat.categories  # index is steamid
unique_app_ids  = df["appid"].astype("category").cat.categories           # index is appid

n_users = df["user_idx"].max() + 1   # total rows in our matrix
n_apps  = df["app_idx"].max() + 1    # total columns in our matrix

#this is just a check
print(f"\nUnique users : {n_users:,}")
print(f"Unique apps  : {n_apps:,}")


# ═════════════════════════════════════════════════════════════════════════════
# 2. COLLABORATIVE FILTERING  (User-User KNN)
# ═════════════════════════════════════════════════════════════════════════════
# so the matrix is going to look like this:
# Each row is a user
# Each column is a game
# And every cell holds a vallue of 1 = liked, -1 = disliked or missing = no review

# A sparse matrix is used because otherwise we are going to get memory issues again? anyway since a lot of users have only reviewed a view of the games
# this type of matrix only stores not zero values.
    
# Convert voted_up (1/0) to explicit +1 / -1 ratings.
# This ensures thumbs-down reviews are stored as -1 (non-zero) in the sparse
# matrix, rather than 0 which would make them invisible and indistinguishable, from "never reviewed".
# Also same reason for the .loc[:, "]
df.loc[:, "rating"] = df["voted_up"].map({1: 1, 0: -1})


#once again just to show it is working on making the matrix aka which step it is at
print("Building user-app sparse matrix ")

# csr_matrix takes three things:
# (values, (row_indices, col_indices))  and  shape
# Each review row contributes one non-zero cell so the 1 (liked) and -1 (disliked). Cells that are absent entirely (true zeros) mean "never reviewed".
# Also I used csr_matrix instead of the coo_matrix the dude used because. Appently coo is faster to build but slower to do things row by row?
# And that is basically how the KNN stuff works so this works faster for that.
user_app_matrix = csr_matrix(
    (df["rating"].values,            # so the 1 or -1.
     (df["user_idx"].values,         # which row (user)
      df["app_idx"].values)),         # which column (app)
    shape=(n_users, n_apps)
)

#Then the fit the model part 
# so metric = cosine means similarity is based on angle between vectors, not magnitude
# and algoritm = brute means compares against every user
model_knn = NearestNeighbors(metric="cosine", algorithm="brute")
# and then fit the matrix with it. Apparenlty this makes it answer quicker for like query time?
model_knn.fit(user_app_matrix)

# Time to announce which step we are at ヽ(￣▽￣)ノ
print("KNN model fitted.")



# Time to actually write the KNN code.... YIPPIE
# I grabbed the nearest 5 for the test. but for the end funtion we have to remove it!
# Also remember user_idx is that cat code version of the user id
def get_similar_users(user_idx: int, n_neighbors: int):
    #I made row it's own thing because in the dudes code it was in the kneighbors thing but I wanted it seprated to see better what is going on where.
    row = user_app_matrix.getrow(user_idx)   # extract this user's row as a sparse vector. (This basically also doesn't store the 0... save the pc's)
    distances, indices = model_knn.kneighbors(row, n_neighbors=n_neighbors)
    similar_idxs = indices.flatten()[1:]     # [1:] skips the user themselves since like they are 0
    # Convert integer indices back to real steam-IDs using our reverse-lookup array. (aka this gives the true ID not the cat one)
    return [unique_user_ids[i] for i in similar_idxs]
    # ^you need the [] around it otherwise it gives you an error.

# a quick test just for the fun. And to see if it works (oh whoops my bed another test works and all that stoof now go poof)
#sample_user_idx = 0
#ample_steamid  = unique_user_ids[sample_user_idx]   # real ID for index 0
#similar_users   = get_similar_users(sample_user_idx)
#print(f"Similar users to steam-ID {sample_steamid}:")
#print(similar_users)
# Yippie it does print things


# ═════════════════════════════════════════════════════════════════════════════
# 3. CONTENT-BASED FILTERING  (TF-IDF + Cosine Similarity)
# ═════════════════════════════════════════════════════════════════════════════
# Okay so the idea behind this one. Smack the two dataframes togheter on the appids.
# Then make all the tags just one long string
# Let TF-IDF lose on those strings. commons tags like "singleplayer" or something really normal get low weight
# But rarer tags like "shooter" (idk what is rare okay so bare with me CS:GO was the first game in the list) get a high weight.
# This makes games with similiar high weight tags closer to eachother then if they would be sharing a low weight one.
# And then grab the cosine similarity from all the tag vectors pairs. The more game overlap in tags the closer to 1 they get.
# Oh btw this uhm... is more my frankenstein of a code then the dudes code since his worked on discription and ours goes for the tags.

# Update time~
print("Transforming all tags to strings")

# OKAY so since our csv had 3 columns worth of tags (likes) we diced to use. we gotta chance plans
# First no clue if there is NaN make it an empty string. No errors or crashes here.
# Then split each column on commas since that is how they are build. Get the tags alone.
# Strip the whitespaces they are probably there. also make everything lowercase!
# Comine all 3 list of tokens into one. Use set so there are not dublicates
# Then we create one long string seperated by spaces so my old code still works!

def split_col(item):
        # Split on the comma in the string so you make into a list of stripped tokens.
        # Returns an empty list if the value is missing. just so no crashes
        if pd.isna(item) or item == "":
            return []
        #turns item into a string then loops through it on t split on the ,
        return [t.strip() for t in str(item).split(",")]

#explenation up there ^
def build_tag_string(row):
    tokens = (
        split_col(row["Categories"]) +
        split_col(row["Tags"]) +
        split_col(row["genre"])
    )
    # set() removes duplicates; sorted() keeps the order
    unique_tokens = sorted(set(tokens))
    #create an string and add to it so it returns the long string
    return " ".join(unique_tokens)

#create another row in the games df with the tag_string. apply like goes over the columns and uses the def created above. axis 1 is columns
games_df["tag_string"] = games_df.apply(build_tag_string, axis=1)

# Now we couple the strings to the app id's in a dataframe ^-^
# Also I made it games with no tags (I think those exist) that just gives an empty string so TF-IDF still produces a row for them... just zeros
app_tag_df = (
    pd.DataFrame({"appid": unique_app_ids}) #makes the dataframe with the unique app ids created in the first part
    .merge(games_df[["appid", "Name", "tag_string"]], on="appid", how="left") #merges the strings in on the left based on appid
    .fillna({"tag_string": "", "Name": "Unknown"})  # games with no tags get an empty document and name is unknown for the end result thingy
)

# Keep a name lookup Series or something. appid linked to game name, for readable output later when we return stuff
appid_to_name = app_tag_df.set_index("appid")["Name"]

# Separate Series of just the tag strings, for TF-IDF (idk how to rewrite the code for the update time and I am too tired for that right now
# So deal with this
app_tag_strings = app_tag_df["tag_string"]

#it is update time (〜^∇^)〜 (it works so notes now
#print(f"Games with tags    : {(app_tag_strings != '').sum():,}") #shows how many games had tags cuz the string isn't empty
#print(f"Games without tags : {(app_tag_strings == '').sum():,}") #Shows how many gmaes had no tags... thus similarity will be 0 cuz there is nothing

#print("Building TF-IDF matrix from game tags")

# So Tfidfectorizer is a scikit-learn thing that coverts reaw thext into a numerical matrix of TF-IDF features.
# So analyzer = word treat each tag as a word token. Not the characters. 
# ngram_range=(1, 2) consider single tags AND pairs of adjacent tags. This lets the model learn that "Open World" together 
# is more meaningful than "Open" and "World" separately.
# Yea I don't get that one a lot either but think it just makes sure that if there was a space in the tag it grabs those words togheter instead of
# Making them two seperate ones?
# min_df=2 is so that it ignores if there is only one tag in the entire dataset. Since well nothing to compare it to so useless information
# also there is a thing you can add called (stop_words="english") but like if there is a stopword in the tags I think it would still be usefull?
# it is not as if we are trying to compare books or something
tf = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2)
tfidf_matrix = tf.fit_transform(app_tag_strings)
# The result is a sparse matrix of shape (n_apps, n_unique_tag_features)

#update time (｢• ω •)｢ wow look how far we already got... okay me this is all me I can't tell if I am losing my mind
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")


# it is callable function time again~ time to find games with similar tags. Oh wow this part I can steal off the dude and I have no idea what it does
# so apparently linear_kernel is equivalent to cosine similarity. But only when the vectors are already "L2-normalised" which is the case in tf-idf vectors
# Anyway the function grabs the cat app ids. And then same as by the other one 5 other neighbours
# And this functions returns a list of unique appids that are the most similiar to the one we had
def get_similar_apps(app_idx: int, n_neighbors: int):
    # Compute similarity between this one app and ALL other game in one go. Cuz apparently we can do that now?
    # Result is a 1D array of similarity scores, one per game
    sims = linear_kernel(tfidf_matrix[app_idx], tfidf_matrix).flatten()

    # So argsort does what it does. It returns indices that would sort the array in ascending order.
    # That is also why we need [::-1] cuz reverses it to descending (highest similarity first).
    # [1:n_neighbors] skips index 0 (the game itself) cuz no we don't need to recommend the game itself you know.
    indices = sims.argsort()[::-1][1:n_neighbors]

    # Convert integer indices back to real appids. aka not the cat version
    return [unique_app_ids[i] for i in indices]
    

    
# Also another quick test time =＾● ⋏ ●＾= (look it is a cat.. it is staring into my soul) (it worked so notes now)
#sample_app_idx = 0
#sample_appid   = unique_app_ids[sample_app_idx]
#similar_apps   = get_similar_apps(sample_app_idx)
#print(f"Apps similar to appid {sample_appid}: {similar_apps}")

# ═════════════════════════════════════════════════════════════════════════════
# 4. HYBRID RECOMMENDER (time to mush those two monsters into one)
# ═════════════════════════════════════════════════════════════════════════════
# So this part grabs both of the 2 and 3. And makes them work togheter.
# This is because each part has it own weakness:
# Collaborative filtering doesn't really recommend games with a lot of reviews. so it won't appear a lot if at all.
# And content based only recommend games simliar to what the user already knows. Not really a lot of variety.
# Mixing both of these we get games recommended from users with a simliar taste. 
# Grab the games those users gave a thumbs up. 
# Then add like games that have simliar tags
# Look how many times a game apears. the more it apears the stronger the recommendation.
# Remove games we know the user has already reviewed. And return the top however many we want!
# That probably sounds easier to do than to make.

# update time! ʘ‿ʘ 
print("now creating the callable feature to recommend games with")
# Now creating the callable feature~ 
# So first input the steam id of the user we want to make a recommendation for (NOT THE CAT VERSION JUST THE LONG STRING)
# Then like the amount of simliar users we want and the simliar apps and how many recommendations we want. You know the good stuff.
# And like I show. I returnes a list of the recommended appids. soreted by hybrid score most recommended first.
def recommend_apps(target_steamid, n_similar_users: int, n_similar_apps: int, top_n: int) -> list:
    # First step. Check if wanted user is even in our system!
    # We have the unique_user_ids. So check if userid in there otherwise throw a hissy fit.
    # Then use .get_loc() to grab convert the steamid into the cat.code stuff so we can use it as a row index in the user-app matrix we made.
    if target_steamid not in unique_user_ids:
        raise ValueError(f"steam-ID {target_steamid} not found in filtered dataset. Please try again")
    # So it only gets to this step if it didn't raise the error. Just so you guys understand the error breaks out of this function. (pretty sure it just
    # ends the program but I would need to test that when Aryan gives me the game dataset with tags and everything works).
    user_idx = unique_user_ids.get_loc(target_steamid)

    # First step after first step! find similar users aka collaborative filtering
    # The plus one is cuz the first one would be the user itself.
    similar_steam_ids = get_similar_users(user_idx, n_neighbors=n_similar_users + 1)

    # Next step! collect games those users liked!
    # create a set to save the data in
    seed_apps = set()
    # then loop through the dataframe finding all the reviews that were positive.
    for sid in similar_steam_ids:
        liked = df[
            (df["author_steamid"] == sid) &
            (df["voted_up"] == 1)
        ]["appid"].unique() #this makes sure it grabs the appid but that it is unique
        seed_apps.update(liked) #then adds it to the set. 

    # Now the next step now we got the liked games. This is where the tags stuff comes in.
    # Candidate_scores tracks how many "votes" each candidate game has received.
    # Every time a game appears as content-similar to a seed, its score goes up by 1.
    # so create the dict that keeps tract of the game and what their score is
    candidate_scores: dict = {}

    # then start looping through the appid's we got from the users liked list (probably a lot of games since it is from all the users and stuff)
    for appid in seed_apps:
        # check if the appid is in our list (just to make sure since we did filter data out)
        if appid not in unique_app_ids:
            continue   # skip if this game wasn't in our filtered data
        # Then we grab the cat version of the appid if was
        a_idx = unique_app_ids.get_loc(appid)
        # Then we get like the games with a lot of simliar tags!
        for sim_appid in get_similar_apps(a_idx, n_neighbors=n_similar_apps + 1):
            #Then we grab the dict and either create a entry and add a point for the game. Or just add a point for the game.
            candidate_scores[sim_appid] = candidate_scores.get(sim_appid, 0) + 1

    # great now we have dict full of games with simliar tags to the games the nearest neighbour users had... Does this still make sense to you guys?
    # I think I am doing pretty well at explaining... and praying this actually works cuz I can't test it.

    # anyway next step is removing games the user knows (∿°○°)∿ 
    # We do this for every game the user knows. So be it a negative or a positive review!
    # So we create a set with all the know steam reviews appid's
    already_reviewed = set(df[df["author_steamid"] == target_steamid]["appid"].unique())
    # Then we loop the dict over this set. Something something oh didn't we have this by networkscience last year. Fun. 
    # anyway this makes it so the dict stuff only saves in the dict again if it isn't already in the set above.
    candidate_scores = {
        k: v for k, v in candidate_scores.items() if k not in already_reviewed
    }

    # now let them battle it out (aka rank them) and return the top however the many
    #sorted does you know. sort. reverse makes sure it is highest on top. And well we sort the dict based on the key.
    ranked = sorted(candidate_scores, key=candidate_scores.get, reverse=True)
    #and then return until the top_n we gave. in a human readable way plus the apid (as int cuz damn) oh whoops forgot the cut off fixed now
    return [(appid_to_name.get(a, "Unknown"), int(a)) for a in ranked[:top_n]]

print("finished set up ready for testing")


Unique users : 2,957,680
Unique apps  : 101,212
Building user-app sparse matrix 
KNN model fitted.
Transforming all tags to strings
TF-IDF matrix shape: (101212, 14128)
now creating the callable feature to recommend games with
finished set up ready for testing


In [13]:
#I am lazy just give me random users from the list to test it with
random_user = df["author_steamid"].sample(1).values[0]
print(f"for user: {random_user}.")
#recommend_apps(target_steamid, n_similar_users: int, n_similar_apps: int, top_n: int) -> list: (so I know what to fill in)
print(recommend_apps(random_user, 10, 10, 10))

#good news! valueerror works. Sad news, I am not in the dataset :( 

for user: 76561198832656235.
[("Tom Clancy's Rainbow Six® Siege X", 488822), ("Tom Clancy's Rainbow Six® Siege X", 488824), ("Tom Clancy's Rainbow Six® Siege X", 488821), ("Tom Clancy's Rainbow Six® Siege X", 488823), ('Apartament 1406: Horror', 2419900), ('DARK SOULS\x99 III', 374320), ('Left 4 Dead 2', 550), ('Saints Row: The Third', 55230), ('BrainBread 2', 346330), ('Survive the Nights', 541300)]


In [22]:
# ═════════════════════════════════════════════════════════════════════════════
# 5. HIT RATE K EVALUATION
# ═════════════════════════════════════════════════════════════════════════════
# How:
#   Sample X users who have ≥10 reviews, just to further desparse the data
#   For each user, hide one positively-reviewed game
#   Rebuild the user-app matrix WITHOUT those hidden reviews (one rebuild)
#   For each test user, get top-25 recommendations
#   A "hit" is the hidden game appears somewhere in those 25 results
#   Hit Rate (25) = hits / total test users

eval_top_k = 25   # how deep into the recommendation list we check
n_similar_users = 25   # neighbours to pull for collaborative filtering
n_similar_apps = 25   # similar games to pull per seed game
min_reviews = 10   # minimum reviews a user needs to be eligible
eval_sample_size = 250 # number of test users

random.seed(42)
np.random.seed(42)

print("HIT RATE K EVALUATION")
# find eligible users, those who have >= min_reviews
print(f"\n[1/6] Finding users with ≥{min_reviews} reviews.")

user_review_counts = df.groupby("author_steamid").size()
eligible_users_all = user_review_counts[user_review_counts >= min_reviews].index

# Also requires at least one positive review to hide
users_with_positives = df[df["voted_up"] == 1]["author_steamid"].unique()
eligible_users = eligible_users_all[eligible_users_all.isin(users_with_positives)]

print(f"Eligible users found : {len(eligible_users):,}")
# this incase there are less eligibile users than the requested sample size
if len(eligible_users) < eval_sample_size:
    print(f"Only {len(eligible_users):,} eligible users found, using all of them.")
    eval_sample_size = len(eligible_users)
# Random sample from the eligible users, as the dataset is somewhat ordered
sampled_users = pd.Index(
    np.random.choice(eligible_users, size=eval_sample_size, replace=False)
)
print(f"Sampled {eval_sample_size:,} test users.")

# Then we build the hold-out set
print("\n[2/6] Building hold-out set, which hides one positive review per user.")
hidden_games = {}   # dict with {steamid: appid of the hidden game}
for steamid in sampled_users:
    # All positively reviewed games for this user
    pos_reviews = df[
        (df["author_steamid"] == steamid) &
        (df["voted_up"] == 1)
    ]["appid"].values
    # Pick one at random to hide
    hidden_appid = np.random.choice(pos_reviews)
    hidden_games[steamid] = hidden_appid
# does this for all steamids
print(f"Hidden one game for each of {len(hidden_games):,} users.")
# Rebuild the user-app matrix without the hidden reviews
# We drop exactly the (user, hidden_game) rows for our test users so the
# recommender cannot "cheat" by seeing those interactions.
print("\n[3/6] Rebuilding user-app matrix without hidden reviews.")

# Builds a boolean mask, True = row we keep
def is_hidden(row):
    return (row["author_steamid"] in hidden_games and
            row["appid"] == hidden_games[row["author_steamid"]])

# We do this because it is faster than .apply() on larger dataframes
hidden_mask = pd.Series(False, index=df.index)
for steamid, appid in hidden_games.items():
    hidden_mask |= ((df["author_steamid"] == steamid) & (df["appid"] == appid))
eval_df = df[~hidden_mask].copy()
print(f"   Original rows : {len(df):,}")
print(f"   Eval rows     : {len(eval_df):,}  ({len(df) - len(eval_df):,} hidden)")

# Re-indexes users and apps on eval_df so the items inside the sparse matrix stay connected and are not bothered by the dropped items
eval_df.loc[:, "user_idx"] = eval_df["author_steamid"].astype("category").cat.codes
eval_df.loc[:, "app_idx"]  = eval_df["appid"].astype("category").cat.codes
eval_unique_user_ids = eval_df["author_steamid"].astype("category").cat.categories
eval_unique_app_ids  = eval_df["appid"].astype("category").cat.categories
eval_n_users = eval_df["user_idx"].max() + 1
eval_n_apps  = eval_df["app_idx"].max() + 1
eval_user_app_matrix = csr_matrix(
    (eval_df["rating"].values,
     (eval_df["user_idx"].values,
      eval_df["app_idx"].values)),
    shape=(eval_n_users, eval_n_apps)
)

print(f"   Eval matrix shape: {eval_user_app_matrix.shape}")

# Now we fit a fresh KNN model on the eval matrix 
print("\n[4/6] Fitting KNN model on eval matrix.")
eval_knn = NearestNeighbors(metric="cosine", algorithm="brute")
eval_knn.fit(eval_user_app_matrix)
print("   KNN fitted.")
# Now to define eval-specific versions of the helper functions used in the main model
# These are identical to the originals but use the eval matrix and index
def eval_get_similar_users(user_idx: int, n_neighbors: int):
    row = eval_user_app_matrix.getrow(user_idx)
    distances, indices = eval_knn.kneighbors(row, n_neighbors=n_neighbors)
    similar_idxs = indices.flatten()[1:]
    return [eval_unique_user_ids[i] for i in similar_idxs]

def eval_get_similar_apps(app_idx: int, n_neighbors: int):
    # TF-IDF matrix is built from ALL the games, as the game tags do not change, so we
    # reuse tfidf_matrix and unique_app_ids from the main code here.
    sims = linear_kernel(tfidf_matrix[app_idx], tfidf_matrix).flatten()
    indices = sims.argsort()[::-1][1:n_neighbors]
    return [unique_app_ids[i] for i in indices]

def eval_recommend_apps(target_steamid, n_similar_users, n_similar_apps, top_n) -> list:
    if target_steamid not in eval_unique_user_ids:
        return [] # checks if the user has no data in the end after the hold out, should not be the case but just in case
    user_idx = eval_unique_user_ids.get_loc(target_steamid)
    similar_steam_ids = eval_get_similar_users(user_idx, n_neighbors=n_similar_users + 1)
    seed_apps = set()
    for sid in similar_steam_ids:
        liked = eval_df[
            (eval_df["author_steamid"] == sid) &
            (eval_df["voted_up"] == 1)
        ]["appid"].unique()
        seed_apps.update(liked)
    candidate_scores: dict = {}
    for appid in seed_apps:
        if appid not in unique_app_ids: # uses full unique_app_ids for TF-IDF
            continue
        a_idx = unique_app_ids.get_loc(appid)
        for sim_appid in eval_get_similar_apps(a_idx, n_neighbors=n_similar_apps + 1):
            candidate_scores[sim_appid] = candidate_scores.get(sim_appid, 0) + 1
    # Remove games the user already reviewed IN THE EVAL DF, which does not include the hidden game of course
    already_reviewed = set(eval_df[eval_df["author_steamid"] == target_steamid]["appid"].unique())
    candidate_scores = {k: v for k, v in candidate_scores.items() if k not in already_reviewed}
    ranked = sorted(candidate_scores, key=candidate_scores.get, reverse=True)
    # return appids only for easy comparison
    return [int(a) for a in ranked[:top_n]] 

# This runs the evaluation
print(f"\n[5/6] Running Hit Rate@{eval_top_k} evaluation on {eval_sample_size} users.")
print("This may take a while given the sample size")

hits= 0
skipped = 0
# keeps a detailed row per user for later inspection
results_log = []   
for i, steamid in enumerate(tqdm(sampled_users, desc="Evaluating users")):
    hidden_appid = hidden_games[steamid]
    # Skips if after removing the hidden review this user has no data at all
    if steamid not in eval_unique_user_ids.values:
        skipped += 1
        continue
    recs = eval_recommend_apps(
        target_steamid  = steamid,
        n_similar_users = n_similar_users,
        n_similar_apps  = n_similar_apps,
        top_n           = eval_top_k
    )
    # edge case, no recommendations generated
    if not recs:    
        skipped += 1
        continue
    is_hit = int(hidden_appid) in recs
    if is_hit:
        hits += 1

    results_log.append({
        "steamid" : steamid,
        "hidden_appid" : hidden_appid,
        "hidden_name" : appid_to_name.get(hidden_appid, "Unknown"),
        "hit" : is_hit,
        "rec_rank" : recs.index(int(hidden_appid)) + 1 if is_hit else None,
        "n_recs" : len(recs)
    })
# This reports the results
evaluated       = eval_sample_size - skipped
hit_rate        = hits / evaluated if evaluated > 0 else 0.0

results_df = pd.DataFrame(results_log)
print(f"HIT RATE @ {eval_top_k} RESULTS")
print(f"  Users sampled : {eval_sample_size:,}")
print(f"  Users skipped : {skipped:,}  (no usable data after hold-out)")
print(f"  Users evaluated : {evaluated:,}")
print(f"  Hits : {hits:,}")
print(f"  Hit Rate @ {eval_top_k} : {hit_rate:.4f}  ({hit_rate * 100:.2f}%)")

if not results_df.empty and results_df["hit"].sum() > 0:
    hit_rows = results_df[results_df["hit"]]
    print(f"\n  Avg rank of hidden game (when hit) : {hit_rows['rec_rank'].mean():.2f}")
    print(f"Median rank of hidden game (hit) : {hit_rows['rec_rank'].median():.0f}")
    print(f"\n  Top 5 example hits:")
    print(hit_rows[["steamid", "hidden_name", "rec_rank"]].head(5).to_string(index=False))

HIT RATE K EVALUATION

[1/6] Finding users with ≥10 reviews.
Eligible users found : 1,915,285
Sampled 250 test users.

[2/6] Building hold-out set, which hides one positive review per user.
Hidden one game for each of 250 users.

[3/6] Rebuilding user-app matrix without hidden reviews.
   Original rows : 48,806,041
   Eval rows     : 48,805,791  (250 hidden)
   Eval matrix shape: (2957680, 101212)

[4/6] Fitting KNN model on eval matrix.
   KNN fitted.

[5/6] Running Hit Rate@25 evaluation on 250 users.
This may take a while given the sample size


Evaluating users: 100%|████████████████████████████████████████████████████████████| 250/250 [1:41:26<00:00, 24.35s/it]


HIT RATE @ 25 RESULTS
  Users sampled          : 250
  Users skipped          : 0  (no usable data after hold-out)
  Users evaluated        : 250
  Hits                   : 11
  Hit Rate @ 25         : 0.0440  (4.40%)

  Avg rank of hidden game (when hit) : 8.18
Median rank of hidden game (hit)   : 10

  Top 5 example hits:
          steamid      hidden_name  rec_rank
76561197999862264 Counter-Strike 2       2.0
76561198239209762        Fallout 4      10.0
76561198008817486        Fallout 4      12.0
76561198964318454 Counter-Strike 2      17.0
76561198172389813  Team Fortress 2      22.0
